# Inference Test — VishingModelWrapper

End-to-end test notebook that loads a trained model and exercises it with synthetic JSONs,
simulating exactly the flow that a production scoring service would receive.

## Flow
1. Load wrapper from S3 (or local disk as fallback)
2. Inspect expected features
3. Build 3 synthetic sessions: legitimate, clear vishing, ambiguous case
4. Get predictions with the three API methods
5. Batch test and error validation
6. Comparative visualization of scores

## 1. Setup

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from io import BytesIO
from urllib.parse import urlparse
from pathlib import Path

print('Libraries loaded.')

## 2. VishingModelWrapper Definition

The class must be defined in this notebook before `joblib` can deserialize the `.pkl` file.

In [ ]:
class VishingModelWrapper:
    """
    Bundles a trained classifier with its scaler, ordered feature list, and
    optimal decision threshold so that inference can be driven purely by a
    named-feature JSON — no positional assumptions required.

    Supports single observations (dict or JSON string) and batches (list of dicts).
    """

    def __init__(self, model, feature_names, scaler=None, threshold=0.5,
                 model_name="", technique="", ratio=""):
        self.model         = model
        self.feature_names = list(feature_names)
        self.scaler        = scaler
        self.threshold     = threshold
        self.model_name    = model_name
        self.technique     = technique
        self.ratio         = ratio

    def _to_array(self, json_input):
        if isinstance(json_input, str):
            data = json.loads(json_input)
        elif isinstance(json_input, dict):
            data = json_input
        elif isinstance(json_input, list):
            return np.vstack([self._to_array(item) for item in json_input])
        else:
            raise TypeError(f"Expected dict, JSON string, or list of dicts. Got {type(json_input)}")

        missing = set(self.feature_names) - set(data.keys())
        if missing:
            raise ValueError(f"Missing features: {sorted(missing)}")

        X = np.array([[data[f] for f in self.feature_names]], dtype=np.float64)
        if self.scaler is not None:
            X = self.scaler.transform(X)
        return X

    def predict(self, json_input):
        """Returns 0 (legitimate) or 1 (vishing) using the stored threshold."""
        proba  = self.model.predict_proba(self._to_array(json_input))[:, 1]
        labels = (proba >= self.threshold).astype(int).tolist()
        return labels[0] if len(labels) == 1 else labels

    def predict_proba_raw(self, json_input):
        """Returns {'legitimate': float, 'vishing': float} per observation."""
        proba = self.model.predict_proba(self._to_array(json_input))
        rows  = [{"legitimate": round(float(p[0]), 6),
                  "vishing":    round(float(p[1]), 6)} for p in proba]
        return rows[0] if len(rows) == 1 else rows

    def predict_full(self, json_input):
        """
        Full inference result: prediction, label, probabilities, threshold_used.
        Accepts dict, JSON string, or list[dict] for batch.
        """
        proba   = self.model.predict_proba(self._to_array(json_input))
        results = []
        for p in proba:
            label = int(p[1] >= self.threshold)
            results.append({
                "prediction":             label,
                "label":                  "vishing" if label == 1 else "legitimate",
                "probability_vishing":    round(float(p[1]), 6),
                "probability_legitimate": round(float(p[0]), 6),
                "threshold_used":         round(self.threshold, 6),
            })
        return results[0] if len(results) == 1 else results

    def __repr__(self):
        return (f"VishingModelWrapper(model={self.model_name!r}, "
                f"technique={self.technique!r}, ratio={self.ratio!r}, "
                f"n_features={len(self.feature_names)}, threshold={self.threshold:.4f})")

print("VishingModelWrapper defined successfully.")


## 3. Load the Wrapper

Set `USE_S3 = True` if you are on SageMaker with access to the bucket.
With `USE_S3 = False` it loads from the local `modelos_temp/` folder generated during training.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
USE_S3 = True   # False → load from local disk

# S3
BUCKET     = 'poc-fraude-vishing'
TECHNIQUE  = 'random_oversampling'         # borderline_smote | random_oversampling | smote | smote_undersampling
RATIO      = '10'            # 10 | 20 | 25
MODEL_NAME = 'XGBoost'       # Logistic_Regression | Random_Forest | XGBoost | Deep_Learning_PyTorch

S3_MODEL_KEY = f'proyecto/modelos/{TECHNIQUE}/{RATIO}/{MODEL_NAME}.pkl'

# Local (fallback)
LOCAL_PATH = Path(f'modelos_temp/{TECHNIQUE}/{RATIO}/{MODEL_NAME}.pkl')

# ── Loading ──────────────────────────────────────────────────────────────────
wrapper = None

if USE_S3:
    try:
        import boto3
        s3 = boto3.client('s3')
        print(f'Downloading from s3://{BUCKET}/{S3_MODEL_KEY} ...')
        with BytesIO() as f:
            s3.download_fileobj(Bucket=BUCKET, Key=S3_MODEL_KEY, Fileobj=f)
            f.seek(0)
            wrapper = joblib.load(f)
        print('Wrapper loaded from S3.')
    except Exception as e:
        print(f'[WARN] S3 not available ({e}). Trying local load...')
        USE_S3 = False

if not USE_S3:
    if LOCAL_PATH.exists():
        wrapper = joblib.load(LOCAL_PATH)
        print(f'Wrapper loaded from disk: {LOCAL_PATH}')
    else:
        raise FileNotFoundError(
            f'Wrapper not found at {LOCAL_PATH}.\n'
            'Run the notebook 7_Modeling_Vishing_AD_AWS_exec.ipynb first.'
        )

print(f'\n{wrapper}')

## 4. Wrapper Inspection

In [ ]:
print('═' * 60)
print(f'  Model       : {wrapper.model_name}')
print(f'  Technique   : {wrapper.technique}')
print(f'  Ratio       : {wrapper.ratio}%')
print(f'  Threshold   : {wrapper.threshold:.4f}')
print(f'  Scaler      : {"StandardScaler" if wrapper.scaler is not None else "Not applicable (tree)"}')
print(f'  N features  : {len(wrapper.feature_names)}')
print('═' * 60)
print()
print('Expected features (in order):')
for i, feat in enumerate(wrapper.feature_names):
    print(f'  [{i:2d}] {feat}')

## 5. Validation with Real Data — S3

Before using synthetic JSONs, we validate that the model discriminates correctly on real
samples extracted from the same dataset it was trained on. This confirms that the wrapper
works correctly end-to-end.

In [ ]:
import boto3

# ── Columns dropped before training (same as in notebook 7) ──────────────────
COLS_TO_DROP = [
    'session_id', 'customer_id', 'session_timestamp',
    'device_type', 'os_type', 'app_version',
    'biocatch_risk_score', 'biocatch_genuine_score', 'biocatch_ato_indicator',
    'biocatch_social_eng_indicator', 'biocatch_bot_indicator',
    'days_to_claim', 'claim_category',
    'screens_visited', 'unusual_screen_visits', 'is_synthetic', 'interactions_per_s',
    'is_vishing',
]

# ── Load a small sample of the parquet from S3 ───────────────────────────────
S3_DATA_KEY = 'proyecto/data/augmented_data/dataset_1M_vishing_.parquet'
N_LEGIT    = 5
N_VISHING  = 5

print(f'Loading sample from s3://{BUCKET}/{S3_DATA_KEY} ...')
s3_data = boto3.client('s3')
with BytesIO() as buf:
    s3_data.download_fileobj(Bucket=BUCKET, Key=S3_DATA_KEY, Fileobj=buf)
    buf.seek(0)
    df_full = pd.read_parquet(buf)

# Split by class and take samples
df_legit   = df_full[df_full['is_vishing'] == 0].sample(N_LEGIT,   random_state=1)
df_vishing = df_full[df_full['is_vishing'] == 1].sample(N_VISHING, random_state=1)
df_sample  = pd.concat([df_legit, df_vishing]).reset_index(drop=True)
y_real     = df_sample['is_vishing'].tolist()

# Apply the same drops as training and keep only the wrapper's features
df_features = df_sample.drop(columns=[c for c in COLS_TO_DROP if c in df_sample.columns])
df_features = df_features[wrapper.feature_names]  # canonical order of the wrapper

print(f'Samples loaded: {N_LEGIT} legitimate + {N_VISHING} vishing\n')

# ── Inference via wrapper (production path) ───────────────────────────────────
records = df_features.to_dict(orient='records')
results_real = wrapper.predict_full(records)

# ── Show results ──────────────────────────────────────────────────────────────
print(f'{"#":<4} {"Actual class":>12}  {"Prediction":>12}  {"P_vishing":>12}  {"Correct"}')
print('─' * 60)
correct = 0
for i, (real, res) in enumerate(zip(y_real, results_real)):
    real_label = 'vishing'   if real == 1 else 'legitimate'
    pred_label = res['label']
    ok         = '✅' if real_label == pred_label else '❌'
    if real_label == pred_label:
        correct += 1
    print(f'{i:<4} {real_label:>12}  {pred_label:>12}  {res["probability_vishing"]:>12.6f}  {ok}')

print(f'\nCorrect: {correct}/{N_LEGIT + N_VISHING}')
print(f'Threshold used: {wrapper.threshold:.4f}')


## 6. Synthetic Sessions — Conceptual Illustration

Three profiles manually built from the ranges of the EDA (`5_EDA_augmented_data.ipynb`).

> **Note:** Models trained on 1M real sessions learn very specific feature combinations.
> The synthetic JSONs illustrate the conceptual patterns of each profile, but they may not
> reach the decision threshold if the values do not exactly replicate the distributions of
> the training set. For functional validation use the previous section (real data).

| Key variable | Legitimate | Vishing | Ambiguous |
|---|---|---|---|
| `phone_call_active` | 0 | 1 | 1 |
| `segmented_typing_ratio` | 0.14 | 0.27 | 0.19 |
| `data_familiarity_score` | 0.68 | 0.53 | 0.62 |
| `hesitation_count` | 4 | 8 | 5 |
| `amount_field_corrections` | 0 | 3 | 1 |
| `remote_access_tool_detected` | 0 | 1 | 0 |
| `transaction_amount_cop` | 150,000 | 9,200,000 | 650,000 |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LEGITIMATE session  — real user, normal behavior
# ─────────────────────────────────────────────────────────────────────────────
session_legitimate = {
    # Keyboard biometrics
    'avg_keyhold_ms'            : 94.5,
    'avg_interkey_latency_ms'   : 145.0,
    'typing_speed_cps'          : 4.75,
    'keystroke_variability'     : 0.22,
    'segmented_typing_ratio'    : 0.14,
    # Touch and swipe
    'avg_touch_pressure'        : 0.52,
    'avg_touch_size_px'         : 45.0,
    'swipe_speed_px_s'          : 382.0,
    'swipe_directional_variance': 0.14,
    'scroll_speed_avg'          : 218.0,
    # Device motion
    'device_tilt_angle_mean'    : 24.5,
    'device_tilt_variability'   : 3.2,
    'gyro_rotation_rate_mean'   : 0.17,
    'accelerometer_jerk_mean'   : 0.11,
    'phone_motion_events'       : 11,
    # Hesitations and pauses
    'hesitation_count'          : 4,
    'avg_hesitation_duration_s' : 0.95,
    'max_hesitation_duration_s' : 2.4,
    'dead_time_periods'         : 3,
    'total_dead_time_s'         : 0.79,
    'dead_time_ratio'           : 0.17,
    # Navigation
    'unique_screens_visited'    : 4,
    'navigation_back_count'     : 2,
    'screen_transition_time_avg_s': 1.8,
    # Errors and corrections
    'input_error_count'         : 1,
    'input_correction_count'    : 2,
    'amount_field_corrections'  : 0,
    'beneficiary_field_corrections': 0,
    'copy_paste_events'         : 0,
    # Familiarity and general behavior
    'data_familiarity_score'    : 0.68,
    'doodling_events'           : 1,
    'session_duration_s'        : 1.0,
    'hour_of_day'               : 14,
    'errors_per_minute'         : 0.0,
    'hesitation_composite'      : 0.0,
    # Risk signals — all off
    'is_atypical_hour'          : 0,
    'phone_call_active'         : 0,
    'call_overlap_duration_s'   : 0.0,
    'remote_access_tool_detected': 0,
    'suspicious_app_detected'   : 0,
    # Usual transaction
    'transaction_attempted'     : 1,
    'transaction_amount_cop'    : 150_000.0,
    'is_new_beneficiary'        : 0,
    'time_to_transaction_s'     : 185.0,
}

# ─────────────────────────────────────────────────────────────────────────────
# VISHING session  — active attack: call + social engineering
# ─────────────────────────────────────────────────────────────────────────────
session_vishing = {
    # Keyboard biometrics — slow and inconsistent under phone instructions
    'avg_keyhold_ms'            : 112.0,
    'avg_interkey_latency_ms'   : 198.0,
    'typing_speed_cps'          : 4.10,
    'keystroke_variability'     : 0.31,
    'segmented_typing_ratio'    : 0.27,
    # Touch and swipe — imprecise under pressure
    'avg_touch_pressure'        : 0.44,
    'avg_touch_size_px'         : 44.5,
    'swipe_speed_px_s'          : 335.0,
    'swipe_directional_variance': 0.26,
    'scroll_speed_avg'          : 192.0,
    # Device motion — more agitated with phone in hand
    'device_tilt_angle_mean'    : 31.2,
    'device_tilt_variability'   : 6.8,
    'gyro_rotation_rate_mean'   : 0.34,
    'accelerometer_jerk_mean'   : 0.22,
    'phone_motion_events'       : 19,
    # Hesitations and pauses — many pauses waiting for instructions
    'hesitation_count'          : 8,
    'avg_hesitation_duration_s' : 1.65,
    'max_hesitation_duration_s' : 6.2,
    'dead_time_periods'         : 7,
    'total_dead_time_s'         : 0.95,
    'dead_time_ratio'           : 0.34,
    # Navigation — similar to the legitimate user to avoid suspicion
    'unique_screens_visited'    : 4,
    'navigation_back_count'     : 3,
    'screen_transition_time_avg_s': 2.3,
    # Errors and corrections — many, modifying transaction data
    'input_error_count'         : 4,
    'input_correction_count'    : 5,
    'amount_field_corrections'  : 3,
    'beneficiary_field_corrections': 3,
    'copy_paste_events'         : 2,
    # Low familiarity — not the user's usual flow
    'data_familiarity_score'    : 0.53,
    'doodling_events'           : 3,
    'session_duration_s'        : 1.0,
    'hour_of_day'               : 22,
    'errors_per_minute'         : 0.11,
    'hesitation_composite'      : 0.18,
    # Risk signals — multiple active
    'is_atypical_hour'          : 1,
    'phone_call_active'         : 1,
    'call_overlap_duration_s'   : 340.0,
    'remote_access_tool_detected': 1,
    'suspicious_app_detected'   : 1,
    # High-amount transaction to an unknown beneficiary
    'transaction_attempted'     : 1,
    'transaction_amount_cop'    : 9_200_000.0,
    'is_new_beneficiary'        : 1,
    'time_to_transaction_s'     : 88.0,
}

# ─────────────────────────────────────────────────────────────────────────────
# AMBIGUOUS session  — mixed signals, genuinely difficult case
# ─────────────────────────────────────────────────────────────────────────────
session_ambiguous = {
    # Keyboard slightly slower than normal
    'avg_keyhold_ms'            : 101.0,
    'avg_interkey_latency_ms'   : 162.0,
    'typing_speed_cps'          : 4.45,
    'keystroke_variability'     : 0.25,
    'segmented_typing_ratio'    : 0.19,
    # Normal touch
    'avg_touch_pressure'        : 0.49,
    'avg_touch_size_px'         : 45.5,
    'swipe_speed_px_s'          : 360.0,
    'swipe_directional_variance': 0.17,
    'scroll_speed_avg'          : 207.0,
    # Normal motion
    'device_tilt_angle_mean'    : 26.8,
    'device_tilt_variability'   : 4.1,
    'gyro_rotation_rate_mean'   : 0.22,
    'accelerometer_jerk_mean'   : 0.14,
    'phone_motion_events'       : 14,
    # Slightly elevated hesitations
    'hesitation_count'          : 5,
    'avg_hesitation_duration_s' : 1.15,
    'max_hesitation_duration_s' : 3.2,
    'dead_time_periods'         : 4,
    'total_dead_time_s'         : 0.83,
    'dead_time_ratio'           : 0.21,
    # Normal navigation
    'unique_screens_visited'    : 5,
    'navigation_back_count'     : 2,
    'screen_transition_time_avg_s': 2.0,
    # Some corrections
    'input_error_count'         : 2,
    'input_correction_count'    : 3,
    'amount_field_corrections'  : 1,
    'beneficiary_field_corrections': 1,
    'copy_paste_events'         : 1,
    # Medium familiarity
    'data_familiarity_score'    : 0.62,
    'doodling_events'           : 2,
    'session_duration_s'        : 1.0,
    'hour_of_day'               : 19,
    'errors_per_minute'         : 0.02,
    'hesitation_composite'      : 0.04,
    # Active call but moderate amount
    'is_atypical_hour'          : 0,
    'phone_call_active'         : 1,
    'call_overlap_duration_s'   : 95.0,
    'remote_access_tool_detected': 0,
    'suspicious_app_detected'   : 0,
    # Moderate transaction to a new beneficiary
    'transaction_attempted'     : 1,
    'transaction_amount_cop'    : 650_000.0,
    'is_new_beneficiary'        : 1,
    'time_to_transaction_s'     : 140.0,
}

cases = {
    'Legitimate': session_legitimate,
    'Vishing'   : session_vishing,
    'Ambiguous' : session_ambiguous,
}

print(f'Sessions defined: {list(cases.keys())}')
print(f'Features in each session: {len(session_legitimate)}')

## 7. Individual Prediction on Synthetic Sessions

In [ ]:
for name, session in cases.items():
    print(f'\n{"═"*55}')
    print(f'  CASE: {name}')
    print(f'{"─"*55}')

    # ── predict() — class 0/1 ─────────────────────────────────────────────
    label_int = wrapper.predict(session)
    print(f'  predict()            → {label_int}  ({"VISHING" if label_int == 1 else "LEGITIMATE"})')

    # ── predict_proba_raw() — raw probabilities ───────────────────────────
    probs = wrapper.predict_proba_raw(session)
    print(f'  predict_proba_raw()  → legitimate={probs["legitimate"]:.6f} | vishing={probs["vishing"]:.6f}')

    # ── predict_full() — full result ─────────────────────────────────────
    full = wrapper.predict_full(session)
    print(f'  predict_full()       →')
    print(json.dumps(full, indent=6))

## 8. Prediction from a JSON String

Simulating the real input from a REST service.

In [ ]:
# Serialize the vishing case as a JSON string (as it would arrive from an API)
payload_json = json.dumps(session_vishing)

print('JSON payload (first 200 characters):')
print(payload_json[:200] + '...')
print()

# The wrapper accepts the JSON string directly
result = wrapper.predict_full(payload_json)

print('Model response:')
print(json.dumps(result, indent=2))

## 9. Batch Prediction — list of dicts

The wrapper accepts a `list[dict]` to process multiple sessions in a single call.

In [ ]:
# List of the 3 cases
batch_input = [session_legitimate, session_vishing, session_ambiguous]
batch_names = ['Legitimate', 'Vishing', 'Ambiguous']

batch_results = wrapper.predict_full(batch_input)


print(f"{'Case':<12}  {'Pred':>5}  {'Label':>12}  {'P_vishing':>12}  {'P_legit':>12}  {'Threshold':>10}")
print('─' * 72)
for name, res in zip(batch_names, batch_results):
    print(
        f'{name:<12}  {res["prediction"]:>5}  {res["label"]:>12}  '
        f'{res["probability_vishing"]:>12.6f}  '
        f'{res["probability_legitimate"]:>12.6f}  '
        f'{res["threshold_used"]:>10.4f}'
    )

## 10. Comparative Visualization of Scores (real data)

In [ ]:
labels_real  = [('vishing' if r == 1 else 'legitimate') for r in y_real]
p_vishing_r  = [r['probability_vishing']    for r in results_real]
p_legit_r    = [r['probability_legitimate'] for r in results_real]
pred_labels  = [r['label']                  for r in results_real]
threshold    = wrapper.threshold

x       = np.arange(len(results_real))
colors  = ['#e74c3c' if real == 'vishing' else '#2ecc71' for real in labels_real]
edges   = ['black' if pred != real else 'none'
           for pred, real in zip(pred_labels, labels_real)]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Vishing score per observation ─────────────────────────────────────────────
bars = axes[0].bar(x, p_vishing_r, color=colors, edgecolor=edges,
                   linewidth=2, width=0.6)
axes[0].axhline(threshold, color='#e67e22', linestyle='--', linewidth=2,
                label=f'Threshold = {threshold:.4f}')
axes[0].set_ylim(0, max(p_vishing_r) * 1.3 + 0.05)
axes[0].set_xticks(x)
axes[0].set_xticklabels(
    [f'#{i}\n({lbl[:3]})' for i, lbl in enumerate(labels_real)], fontsize=9)
axes[0].set_ylabel('P(vishing)', fontsize=12)
axes[0].set_title(
    f'Vishing Score — Real Data\n{wrapper.model_name} | {wrapper.technique} {wrapper.ratio}%',
    fontweight='bold')
axes[0].legend(fontsize=9)
for bar, val in zip(bars, p_vishing_r):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + max(p_vishing_r) * 0.02,
                 f'{val:.4f}', ha='center', fontsize=8, fontweight='bold')

import matplotlib.patches as mpatches
axes[0].legend(handles=[
    axes[0].get_lines()[0],
    mpatches.Patch(color='#e74c3c', label='Actual class: vishing'),
    mpatches.Patch(color='#2ecc71', label='Actual class: legitimate'),
], fontsize=9)

# ── Stacked probabilities ─────────────────────────────────────────────────────
axes[1].bar(x, p_legit_r, label='P(legitimate)', color='#2ecc71', alpha=0.85, width=0.6)
axes[1].bar(x, p_vishing_r, bottom=p_legit_r, label='P(vishing)',
            color='#e74c3c', alpha=0.85, width=0.6)
axes[1].set_xticks(x)
axes[1].set_xticklabels(
    [f'#{i}\n({lbl[:3]})' for i, lbl in enumerate(labels_real)], fontsize=9)
axes[1].set_ylim(0, 1.1)
axes[1].set_ylabel('Probability', fontsize=12)
axes[1].set_title('Probability Distribution (stacked)', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].axhline(1.0, color='gray', linestyle=':', linewidth=1)

plt.tight_layout()
plt.show()

print(f'Green = actual legitimate  |  Red = actual vishing  |  Black edge = misclassification')
print(f'Threshold: {threshold:.4f}  — above → VISHING')


## 9. Comparative Risk Profile — Radar

In [ ]:
# Most discriminative features according to the EDA (manual top-10 selection)
RISK_FEATURES = [
    'segmented_typing_ratio',
    'hesitation_count',
    'data_familiarity_score',
    'amount_field_corrections',
    'phone_call_active',
    'remote_access_tool_detected',
    'beneficiary_field_corrections',
    'keystroke_variability',
    'is_new_beneficiary',
    'errors_per_minute',
]

# EDA ranges to normalize to [0, 1]
RANGES = {
    'segmented_typing_ratio'    : (0.0, 0.76),
    'hesitation_count'          : (0,   15),
    'data_familiarity_score'    : (0.03, 1.0),
    'amount_field_corrections'  : (0,   6),
    'phone_call_active'         : (0,   1),
    'remote_access_tool_detected': (0,  1),
    'beneficiary_field_corrections': (0, 6),
    'keystroke_variability'     : (0.0, 0.66),
    'is_new_beneficiary'        : (0,   1),
    'errors_per_minute'         : (0.0, 0.15),
}
# For data_familiarity_score the risk is INVERSE (higher score → less risk)
INVERTED = {'data_familiarity_score'}

def normalise(feat, val):
    lo, hi = RANGES[feat]
    norm = (val - lo) / (hi - lo) if hi > lo else 0.0
    norm = max(0.0, min(1.0, norm))
    return 1 - norm if feat in INVERTED else norm

radar_colors = {'Legitimate': '#2ecc71', 'Vishing': '#e74c3c', 'Ambiguous': '#f39c12'}
angles = np.linspace(0, 2 * np.pi, len(RISK_FEATURES), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

for name, session in cases.items():
    vals = [normalise(f, session[f]) for f in RISK_FEATURES] + [normalise(RISK_FEATURES[0], session[RISK_FEATURES[0]])]
    ax.plot(angles, vals, 'o-', color=radar_colors[name], linewidth=2,
            label=name, markersize=6)
    ax.fill(angles, vals, alpha=0.08, color=radar_colors[name])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(RISK_FEATURES, fontsize=9)
ax.set_ylim(0, 1)
ax.set_title('Comparative Risk Profile\n(normalized features, higher = higher risk)',
             fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=11)
plt.tight_layout()
plt.show()

## 10. Validations and Error Handling

In [ ]:
print('=== TEST 1: Missing feature ===')
incomplete_input = {k: v for k, v in session_vishing.items() if k != 'phone_call_active'}
try:
    wrapper.predict_full(incomplete_input)
    print('  ❌ It should have failed')
except ValueError as e:
    print(f'  ✅ Error correctly caught: {e}')

print()
print('=== TEST 2: Invalid JSON string ===')
try:
    wrapper.predict_full('{"wrong_field": 999}')
    print('  ❌ It should have failed')
except ValueError as e:
    print(f'  ✅ Error caught: missing {len(wrapper.feature_names) - 1} features (only the first is shown):')
    print(f'     {str(e)[:120]}...')

print()
print('=== TEST 3: Input with extra features (should be ignored if no error) ===')
# The wrapper filters by feature_names; the extras are simply not used
input_with_extras = {**session_legitimate, 'extra_field': 99999, 'another_field': 'abc'}
try:
    res = wrapper.predict_full(input_with_extras)
    print(f'  ✅ Successful prediction ignoring extra fields → label={res["label"]}')
except Exception as e:
    print(f'  Result: {e}')

print()
print('=== TEST 4: Incorrect input type ===')
try:
    wrapper.predict_full(12345)
    print('  ❌ It should have failed')
except TypeError as e:
    print(f'  ✅ TypeError caught: {e}')

## 11. Final Summary

In [ ]:
print('═' * 65)
print('  INFERENCE SUMMARY')
print('═' * 65)
print(f'  Model       : {wrapper.model_name}')
print(f'  Technique   : {wrapper.technique}  |  Ratio: {wrapper.ratio}%')
print(f'  Threshold   : {wrapper.threshold:.4f}')
print(f'  N features  : {len(wrapper.feature_names)}')
print('─' * 65)
print(f'  {"Case":<12}  {"Vishing score":>14}  {"Decision":>12}')
print('─' * 65)
for name, res in zip(batch_names, batch_results):
    flag = '⚠️  ALERT' if res['prediction'] == 1 else '✅  OK'
    print(f'  {name:<12}  {res["probability_vishing"]:>14.6f}  {flag}')
print('═' * 65)
print()
print('API available on the wrapper:')
print('  wrapper.predict(json)           → int (0 or 1)')
print('  wrapper.predict_proba_raw(json) → dict {legitimate, vishing}')
print('  wrapper.predict_full(json)      → full dict')
print('  All accept: dict | JSON str | list[dict] for batch')